In [30]:
import matplotlib.pyplot as plt
import torch
import torchvision
import os
from torch import nn
from torchvision import transforms, datasets
from torchinfo import summary
from torch.utils.data import DataLoader, Subset

In [59]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = max(1, os.cpu_count() or 1)
IMG_SIZE = 224
SEED = 42
BATCH_SIZE = 32


In [55]:
DEVICE, NUM_WORKERS

('cuda', <function posix.cpu_count()>)

In [39]:
def to_rgb(image):
    return img.convert("RGB")

In [40]:
train_tf = transforms.Compose([
    transforms.Lambda(to_rgb),
    transforms.RandomResizedCrop(IMG_SIZE,scale=(0.65,1.0),ratio=(0.75,1.33)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2,saturation=0.2,hue=0.05, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

In [41]:
test_tf = transforms.Compose([
    transforms.Lambda(to_rgb),
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

In [42]:
from torch.utils.data import random_split 
root_dir = "./data"
test_ratio = 0.2

In [43]:
ds_train_full = datasets.Caltech101(root=root_dir,download=True,transform=train_tf)
ds_test_full  = datasets.Caltech101(root=root_dir,download=True,transform=test_tf)

In [44]:
n_total = len(ds_train_full)
n_test = int(n_total * test_ratio)
n_train = n_total - n_test

In [45]:
n_total, n_train, n_test

(8677, 6942, 1735)

In [46]:
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(n_total, generator=g).tolist()

In [47]:
train_idx = perm[:n_train]
test_idx  = perm[:n_test]

In [48]:
len(train_idx), len(test_idx)

(6942, 1735)

In [49]:
train_ds  = Subset(ds_train_full,train_idx)
test_ds   = Subset(ds_test_full,test_idx)

In [50]:
len(train_ds), len(test_ds)

(6942, 1735)

In [ ]:
train_dataloader = DataLoader(
    dataset=test_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_dataloader = DataLoader(
    dataset=test_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [64]:
len(ds_train_full.categories)

101

In [65]:
class_names = ds_train_full.categories

In [68]:
height = 224
width = 224
color_channel = 3
patch_size = 16

number_of_patches = int((height * width) / (patch_size**2))
number_of_patches

196

In [71]:
embedding_layer_input_shape = (height,width,color_channel)
embedding_layer_output_shape = number_of_patches* (patch_size**2 * color_channel)

In [72]:
print(embedding_layer_input_shape)
print(embedding_layer_output_shape)

(224, 224, 3)
150528
